# Laboratorio 4: Redes Neuronales Convolucionales (CNN)
## Clasificación MNIST y Perros vs Gatos
**Nombre:** ApellidoNombre  
**Fecha:** 2025

> ✅ **Versión Kaggle** — rutas `/kaggle/input/` · gestión de memoria · correcciones de RAM

---
# 0. SETUP GENERAL

In [2]:
# En Kaggle NO se necesita montar Drive
# Tu dataset está en: /kaggle/input/perros-gatos/

import os

# Verificar estructura del dataset
DATASET_ROOT = '/kaggle/input/perros-gatos'
for root, dirs, files in os.walk(DATASET_ROOT):
    level = root.replace(DATASET_ROOT, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for d in dirs:
            print(f'{indent}  {d}/')

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, Model
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import gc

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, f1_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

2026-05-02 14:02:36.704129: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777730556.894413      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777730556.954507      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777730557.403154      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777730557.403199      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777730557.403202      57 computation_placer.cc:177] computation placer alr

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# ── Función central de limpieza de memoria ────────────────────────────────────
def liberar_memoria(*objetos):
    for obj in objetos:
        try: del obj
        except: pass
    keras.backend.clear_session()
    gc.collect()

print('liberar_memoria() lista ✓')

liberar_memoria() lista ✓


In [5]:
# ── Utilidades compartidas ────────────────────────────────────────────────────
def compute_metrics(model, x, y):
    y_pred = np.argmax(model.predict(x, verbose=0), axis=1)
    return (round(accuracy_score(y, y_pred), 4),
            round(precision_score(y, y_pred, average='macro', zero_division=0), 4),
            round(f1_score(y, y_pred, average='macro', zero_division=0), 4))

def compute_metrics_gen(model, generator):
    generator.reset()
    y_prob = model.predict(generator, verbose=0).flatten()
    y_pred = (y_prob > 0.5).astype(int)
    y_true = generator.classes
    return (round(accuracy_score(y_true, y_pred), 4),
            round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
            round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4))

def plot_history(history, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3))
    ax1.plot(history.history['accuracy'],     label='Train')
    ax1.plot(history.history['val_accuracy'], label='Val')
    ax1.set_title(f'{title} — Accuracy'); ax1.legend(); ax1.grid(True)
    ax2.plot(history.history['loss'],     label='Train')
    ax2.plot(history.history['val_loss'], label='Val')
    ax2.set_title(f'{title} — Loss'); ax2.legend(); ax2.grid(True)
    plt.tight_layout(); plt.show(); plt.close(fig); del fig

print('Utilidades listas ✓')

Utilidades listas ✓


---
# ═══════════════════════════════
# PARTE A — MNIST
# ═══════════════════════════════

In [6]:
# ── Carga y preparación MNIST ─────────────────────────────────────────────────
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train_full = (x_train_full.astype('float32') / 255.0)[..., np.newaxis]
x_test       = (x_test.astype('float32')       / 255.0)[..., np.newaxis]

VAL_SIZE = 12000
x_val,   y_val   = x_train_full[:VAL_SIZE],  y_train_full[:VAL_SIZE]
x_train, y_train = x_train_full[VAL_SIZE:],  y_train_full[VAL_SIZE:]

print(f'Train {x_train.shape} | Val {x_val.shape} | Test {x_test.shape}')

# Versión 32×32 RGB para modelos preentrenados
# Se genera bajo demanda dentro del loop para no ocupar RAM permanente
def a_rgb32(x):
    return tf.repeat(tf.image.resize(x, [32, 32]), 3, axis=-1).numpy()

# Muestra visual
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray')
    ax.set_title(str(y_train[i])); ax.axis('off')
plt.suptitle('Muestras MNIST'); plt.tight_layout(); plt.show()
plt.close(); del fig

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train (48000, 28, 28, 1) | Val (12000, 28, 28, 1) | Test (10000, 28, 28, 1)


---
## SECCIÓN 2 — CNN desde cero · MNIST

In [7]:
def build_cnn_scratch(input_shape=(28,28,1), num_classes=10, l2=1e-4):
    reg = regularizers.l2(l2)
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32,(3,3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Conv2D(32,(3,3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.MaxPooling2D(), layers.Dropout(0.25),
        layers.Conv2D(64,(3,3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Conv2D(64,(3,3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.MaxPooling2D(), layers.Dropout(0.25),
        layers.Conv2D(128,(3,3),activation='relu', padding='same', kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu', kernel_regularizer=reg),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='CNN_Scratch')

build_cnn_scratch().summary()

I0000 00:00:1777730583.703495      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "CNN_Scratch"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 175,338 (684.91 KB)

 Trainable params: 174,890 (683.16 KB)

 Non-trainable params: 448 (1.75 KB)

In [8]:
SCRATCH_EXPS = [
    ('Adam',    1e-2), ('Adam',    1e-3), ('Adam',    1e-4),
    ('SGD',     1e-2), ('SGD',     1e-3), ('SGD',     1e-4),
    ('RMSprop', 1e-3), ('RMSprop', 1e-4),  # optimizador adicional
]

results_scratch = []

for opt_name, lr in SCRATCH_EXPS:
    print(f'\n▶ Scratch MNIST | {opt_name} lr={lr}')

    opt = (keras.optimizers.Adam(lr)            if opt_name == 'Adam'
      else keras.optimizers.SGD(lr,momentum=0.9) if opt_name == 'SGD'
      else keras.optimizers.RMSprop(lr))

    model = build_cnn_scratch()
    model.compile(optimizer=opt,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    history = model.fit(x_train, y_train,
                        validation_data=(x_val, y_val),
                        epochs=30, batch_size=128,
                        callbacks=[es], verbose=0)

    plot_history(history, f'Scratch {opt_name} lr={lr}')

    acc_tr, prec_tr, f1_tr = compute_metrics(model, x_train, y_train)
    acc_v,  prec_v,  f1_v  = compute_metrics(model, x_val,   y_val)
    acc_te, prec_te, f1_te = compute_metrics(model, x_test,  y_test)

    results_scratch.append({
        'Optimizador': opt_name, 'LR': lr,
        'Acc_Train': acc_tr, 'Acc_Val': acc_v,  'Acc_Test': acc_te,
        'Prec_Train':prec_tr,'Prec_Val':prec_v, 'Prec_Test':prec_te,
        'F1_Train':  f1_tr,  'F1_Val':  f1_v,   'F1_Test':  f1_te
    })
    print(f'   Test → Acc={acc_te}  F1={f1_te}')

    liberar_memoria(model, history, opt, es)

df_scratch = pd.DataFrame(results_scratch)
print('\n═══ TABLA 1 — CNN SCRATCH — MNIST ═══')
display(df_scratch)

best = df_scratch.loc[df_scratch['Acc_Test'].idxmax()]
print('\n🏆 Mejor:', best[['Optimizador','LR','Acc_Test','F1_Test']].to_dict())


▶ Scratch MNIST | Adam lr=0.01


I0000 00:00:1777730588.785544     127 service.cc:152] XLA service 0x7cf4a4015a70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777730588.785579     127 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1777730589.451723     127 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777730594.982648     127 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   Test → Acc=0.9828  F1=0.9826

▶ Scratch MNIST | Adam lr=0.001
   Test → Acc=0.994  F1=0.994

▶ Scratch MNIST | Adam lr=0.0001
   Test → Acc=0.9942  F1=0.9942

▶ Scratch MNIST | SGD lr=0.01
   Test → Acc=0.9955  F1=0.9955

▶ Scratch MNIST | SGD lr=0.001
   Test → Acc=0.9788  F1=0.9786

▶ Scratch MNIST | SGD lr=0.0001
   Test → Acc=0.8988  F1=0.8974

▶ Scratch MNIST | RMSprop lr=0.001
   Test → Acc=0.9935  F1=0.9935

▶ Scratch MNIST | RMSprop lr=0.0001
   Test → Acc=0.9925  F1=0.9924

═══ TABLA 1 — CNN SCRATCH — MNIST ═══


,Optimizador,LR,Acc_Train,Acc_Val,Acc_Test,Prec_Train,Prec_Val,Prec_Test,F1_Train,F1_Val,F1_Test
0,Adam,0.0100,0.9837,0.9842,0.9828,0.9839,0.9844,0.9831,0.9836,0.9842,0.9826
1,Adam,0.0010,0.9970,0.9936,0.9940,0.9970,0.9936,0.9941,0.9969,0.9935,0.9940
2,Adam,0.0001,0.9966,0.9910,0.9942,0.9966,0.9909,0.9943,0.9966,0.9909,0.9942
3,SGD,0.0100,0.9986,0.9927,0.9955,0.9986,0.9925,0.9955,0.9986,0.9926,0.9955
4,SGD,0.0010,0.9784,0.9738,0.9788,0.9785,0.9738,0.9790,0.9782,0.9735,0.9786
5,SGD,0.0001,0.8948,0.8934,0.8988,0.9000,0.8997,0.9032,0.8935,0.8923,0.8974
6,RMSprop,0.0010,0.9972,0.9930,0.9935,0.9972,0.9930,0.9935,0.9972,0.9930,0.9935
7,RMSprop,0.0001,0.9966,0.9919,0.9925,0.9966,0.9918,0.9926,0.9966,0.9918,0.9924



🏆 Mejor: {'Optimizador': 'SGD', 'LR': 0.01, 'Acc_Test': 0.9955, 'F1_Test': 0.9955}


In [9]:
# 💾 Guardar resultados en /kaggle/working/ (persisten al final del notebook)
df_scratch.to_csv('/kaggle/working/resultados_scratch_mnist.csv', index=False)
print('Guardado ✓')

Guardado ✓


---
## SECCIÓN 3 — Transfer Learning · MNIST

In [10]:
def build_tl_mnist(base_name, l2=1e-4):
    reg = regularizers.l2(l2)
    kw  = dict(weights='imagenet', include_top=False, input_shape=(32,32,3))
    base = {'VGG16':VGG16,'ResNet50':ResNet50,'MobileNetV2':MobileNetV2}[base_name](**kw)
    base.trainable = False
    inp = keras.Input(shape=(32,32,3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu', kernel_regularizer=reg)(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(10, activation='softmax')(x)
    return Model(inp, out, name=f'TL_{base_name}')


TL_EXPS = [(b,o,lr)
           for b  in ['VGG16','ResNet50','MobileNetV2']
           for o  in ['Adam','SGD']
           for lr in [1e-2, 1e-3, 1e-4]]

results_tl = []

for base_name, opt_name, lr in TL_EXPS:
    print(f'\n▶ TL MNIST | {base_name} {opt_name} lr={lr}')

    # Convertir a 32x32 RGB dentro del loop → se libera al final
    print('  Convirtiendo imágenes...')
    x_tr32 = a_rgb32(x_train)
    x_v32  = a_rgb32(x_val)
    x_te32 = a_rgb32(x_test)

    opt   = keras.optimizers.Adam(lr) if opt_name=='Adam' else keras.optimizers.SGD(lr,momentum=0.9)
    model = build_tl_mnist(base_name)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    history = model.fit(x_tr32, y_train,
                        validation_data=(x_v32, y_val),
                        epochs=20, batch_size=128,
                        callbacks=[es], verbose=0)

    plot_history(history, f'TL {base_name} {opt_name} lr={lr}')

    acc_tr,prec_tr,f1_tr = compute_metrics(model, x_tr32, y_train)
    acc_v, prec_v, f1_v  = compute_metrics(model, x_v32,  y_val)
    acc_te,prec_te,f1_te = compute_metrics(model, x_te32, y_test)

    results_tl.append({
        'Modelo':base_name,'Optimizador':opt_name,'LR':lr,
        'Acc_Train':acc_tr,'Acc_Val':acc_v,'Acc_Test':acc_te,
        'Prec_Train':prec_tr,'Prec_Val':prec_v,'Prec_Test':prec_te,
        'F1_Train':f1_tr,'F1_Val':f1_v,'F1_Test':f1_te
    })
    print(f'   Test → Acc={acc_te}  F1={f1_te}')

    liberar_memoria(model, history, opt, es, x_tr32, x_v32, x_te32)

df_tl = pd.DataFrame(results_tl)
print('\n═══ TABLA 2 — TRANSFER LEARNING — MNIST ═══')
display(df_tl)


▶ TL MNIST | VGG16 Adam lr=0.01
  Convirtiendo imágenes...
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
   Test → Acc=0.9593  F1=0.9588

▶ TL MNIST | VGG16 Adam lr=0.001
  Convirtiendo imágenes...
   Test → Acc=0.9746  F1=0.9742

▶ TL MNIST | VGG16 Adam lr=0.0001
  Convirtiendo imágenes...
   Test → Acc=0.9578  F1=0.9572

▶ TL MNIST | VGG16 SGD lr=0.01
  Convirtiendo imágenes...
   Test → Acc=0.9633  F1=0.9629

▶ TL MNIST | VGG16 SGD lr=0.001
  Convirtiendo imágenes...
   Test → Acc=0.9276  F1=0.9265

▶ TL MNIST | VGG16 SGD lr=0.0001
  Convirtiendo imágenes...
   Test → Acc=0.7813  F1=0.7711

▶ TL MNIST | ResNet50 Adam lr=0.01
  Convirtiendo imágenes...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
   Test → Acc=0.9182  F1=0.9168

▶ TL MNIST | ResNet50 Adam lr=0.001
  Convirtiendo imágenes...
   Test → Acc=0.9416  F1=0.9408

▶ TL MNIST | ResNet50 Adam lr=0.0001
  Convirtiendo imágenes...
   Test → Acc=0.9298  F1=0.9288

▶ TL MNIST | ResNet50 SGD lr=0.01
  Convirtiendo imáge

,Modelo,Optimizador,LR,Acc_Train,Acc_Val,Acc_Test,Prec_Train,Prec_Val,Prec_Test,F1_Train,F1_Val,F1_Test
0,VGG16,Adam,0.0100,0.9591,0.9517,0.9593,0.9591,0.9514,0.9594,0.9587,0.9509,0.9588
1,VGG16,Adam,0.0010,0.9778,0.9700,0.9746,0.9776,0.9694,0.9743,0.9776,0.9694,0.9742
2,VGG16,Adam,0.0001,0.9576,0.9515,0.9578,0.9570,0.9506,0.9572,0.9570,0.9505,0.9572
3,VGG16,SGD,0.0100,0.9621,0.9588,0.9633,0.9618,0.9582,0.9632,0.9616,0.9580,0.9629
4,VGG16,SGD,0.0010,0.9237,0.9193,0.9276,0.9226,0.9177,0.9265,0.9226,0.9176,0.9265
5,VGG16,SGD,0.0001,0.7714,0.7742,0.7813,0.7694,0.7726,0.7814,0.7608,0.7632,0.7711
6,ResNet50,Adam,0.0100,0.9090,0.9067,0.9182,0.9081,0.9051,0.9173,0.9077,0.9047,0.9168
7,ResNet50,Adam,0.0010,0.9340,0.9299,0.9416,0.9346,0.9303,0.9415,0.9333,0.9288,0.9408
8,ResNet50,Adam,0.0001,0.9217,0.9193,0.9298,0.9221,0.9190,0.9297,0.9207,0.9179,0.9288
9,ResNet50,SGD,0.0100,0.6289,0.6360,0.6415,0.6014,0.5976,0.6065,0.5927,0.5962,0.6068


In [11]:
df_tl.to_csv('/kaggle/working/resultados_tl_mnist.csv', index=False)
print('Guardado ✓')

Guardado ✓


---
## SECCIÓN 4 — Fine-Tuning · MNIST

In [12]:
def build_ft_mnist(base_name, unfreeze=2, l2=1e-4):
    reg = regularizers.l2(l2)
    kw  = dict(weights='imagenet', include_top=False, input_shape=(32,32,3))
    base = {'VGG16':VGG16,'ResNet50':ResNet50,'MobileNetV2':MobileNetV2}[base_name](**kw)
    base.trainable = True
    for layer in base.layers[:-unfreeze]:
        layer.trainable = False
    inp = keras.Input(shape=(32,32,3))
    x   = base(inp, training=True)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu', kernel_regularizer=reg)(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(10, activation='softmax')(x)
    return Model(inp, out, name=f'FT_{base_name}')


FT_EXPS = [(b,o,lr)
           for b  in ['VGG16','ResNet50','MobileNetV2']
           for o  in ['Adam','SGD']
           for lr in [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]]

results_ft = []

for base_name, opt_name, lr in FT_EXPS:
    print(f'\n▶ FT MNIST | {base_name} {opt_name} lr={lr}')

    print('  Convirtiendo imágenes...')
    x_tr32 = a_rgb32(x_train)
    x_v32  = a_rgb32(x_val)
    x_te32 = a_rgb32(x_test)

    opt   = keras.optimizers.Adam(lr) if opt_name=='Adam' else keras.optimizers.SGD(lr,momentum=0.9)
    model = build_ft_mnist(base_name)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    history = model.fit(x_tr32, y_train,
                        validation_data=(x_v32, y_val),
                        epochs=20, batch_size=128,
                        callbacks=[es], verbose=0)

    plot_history(history, f'FT {base_name} {opt_name} lr={lr}')

    acc_tr,prec_tr,f1_tr = compute_metrics(model, x_tr32, y_train)
    acc_v, prec_v, f1_v  = compute_metrics(model, x_v32,  y_val)
    acc_te,prec_te,f1_te = compute_metrics(model, x_te32, y_test)

    results_ft.append({
        'Modelo':base_name,'Optimizador':opt_name,'LR':lr,
        'Acc_Train':acc_tr,'Acc_Val':acc_v,'Acc_Test':acc_te,
        'Prec_Train':prec_tr,'Prec_Val':prec_v,'Prec_Test':prec_te,
        'F1_Train':f1_tr,'F1_Val':f1_v,'F1_Test':f1_te
    })
    print(f'   Test → Acc={acc_te}  F1={f1_te}')

    liberar_memoria(model, history, opt, es, x_tr32, x_v32, x_te32)

df_ft = pd.DataFrame(results_ft)
print('\n═══ TABLA 3 — FINE-TUNING — MNIST ═══')
display(df_ft)


▶ FT MNIST | VGG16 Adam lr=0.01
  Convirtiendo imágenes...
   Test → Acc=0.9838  F1=0.9837

▶ FT MNIST | VGG16 Adam lr=0.001
  Convirtiendo imágenes...
   Test → Acc=0.9847  F1=0.9846

▶ FT MNIST | VGG16 Adam lr=0.0001
  Convirtiendo imágenes...
   Test → Acc=0.9867  F1=0.9865

▶ FT MNIST | VGG16 Adam lr=1e-05
  Convirtiendo imágenes...
   Test → Acc=0.9769  F1=0.9767

▶ FT MNIST | VGG16 Adam lr=1e-06
  Convirtiendo imágenes...
   Test → Acc=0.9111  F1=0.9095

▶ FT MNIST | VGG16 SGD lr=0.01
  Convirtiendo imágenes...
   Test → Acc=0.9842  F1=0.984

▶ FT MNIST | VGG16 SGD lr=0.001
  Convirtiendo imágenes...
   Test → Acc=0.982  F1=0.9819

▶ FT MNIST | VGG16 SGD lr=0.0001
  Convirtiendo imágenes...
   Test → Acc=0.9568  F1=0.9563

▶ FT MNIST | VGG16 SGD lr=1e-05
  Convirtiendo imágenes...
   Test → Acc=0.8343  F1=0.8277

▶ FT MNIST | VGG16 SGD lr=1e-06
  Convirtiendo imágenes...
   Test → Acc=0.5385  F1=0.4892

▶ FT MNIST | ResNet50 Adam lr=0.01
  Convirtiendo imágenes...
   Test → Acc=

,Modelo,Optimizador,LR,Acc_Train,Acc_Val,Acc_Test,Prec_Train,Prec_Val,Prec_Test,F1_Train,F1_Val,F1_Test
0,VGG16,Adam,0.010000,0.9900,0.9822,0.9838,0.9900,0.9822,0.9839,0.9899,0.9821,0.9837
1,VGG16,Adam,0.001000,0.9906,0.9826,0.9847,0.9906,0.9825,0.9848,0.9905,0.9824,0.9846
2,VGG16,Adam,0.000100,0.9935,0.9844,0.9867,0.9935,0.9841,0.9867,0.9934,0.9841,0.9865
3,VGG16,Adam,0.000010,0.9797,0.9728,0.9769,0.9796,0.9725,0.9769,0.9795,0.9724,0.9767
4,VGG16,Adam,0.000001,0.9017,0.9050,0.9111,0.9018,0.9049,0.9111,0.9002,0.9033,0.9095
5,VGG16,SGD,0.010000,0.9882,0.9813,0.9842,0.9881,0.9810,0.9841,0.9881,0.9810,0.9840
6,VGG16,SGD,0.001000,0.9845,0.9795,0.9820,0.9844,0.9791,0.9819,0.9843,0.9792,0.9819
7,VGG16,SGD,0.000100,0.9545,0.9503,0.9568,0.9541,0.9496,0.9562,0.9541,0.9496,0.9563
8,VGG16,SGD,0.000010,0.8224,0.8257,0.8343,0.8256,0.8278,0.8377,0.8150,0.8178,0.8277
9,VGG16,SGD,0.000001,0.5317,0.5413,0.5385,0.5143,0.5213,0.5090,0.4829,0.4892,0.4892


In [13]:
df_ft.to_csv('/kaggle/working/resultados_ft_mnist.csv', index=False)
print('Guardado ✓')

Guardado ✓


---
## SECCIÓN 5 — Extracción de características + SVM · MNIST

In [14]:
def extraer_features_mnist(base_name):
    print(f'  Convirtiendo imágenes para {base_name}...')
    x_tr32 = a_rgb32(x_train)
    x_v32  = a_rgb32(x_val)
    x_te32 = a_rgb32(x_test)

    kw  = dict(weights='imagenet', include_top=False, input_shape=(32,32,3), pooling='avg')
    ext = {'VGG16':VGG16,'ResNet50':ResNet50,'MobileNetV2':MobileNetV2}[base_name](**kw)

    print(f'  Extrayendo features...')
    ftr = ext.predict(x_tr32, batch_size=256, verbose=0)
    fv  = ext.predict(x_v32,  batch_size=256, verbose=0)
    fte = ext.predict(x_te32, batch_size=256, verbose=0)
    print(f'  Shape: {ftr.shape}')

    liberar_memoria(ext, x_tr32, x_v32, x_te32)

    sc = StandardScaler()
    return sc.fit_transform(ftr), sc.transform(fv), sc.transform(fte)


results_svm = []

for base_name in ['VGG16', 'ResNet50']:
    ftr, fv, fte = extraer_features_mnist(base_name)

    for kernel, grid in [
        ('linear', {'C': [0.01, 0.1, 1, 10]}),
        ('rbf',    {'C': [0.1, 1, 10], 'gamma': ['scale','auto',0.001,0.01]})
    ]:
        print(f'\n  ▶ SVM {kernel} | {base_name}')
        clf = GridSearchCV(SVC(kernel=kernel), grid, cv=3, n_jobs=-1, verbose=0)
        clf.fit(ftr, y_train)
        best = clf.best_estimator_
        print(f'  Params óptimos: {clf.best_params_}')

        def sm(X, yt):
            yp = best.predict(X)
            return (round(accuracy_score(yt,yp),4),
                    round(precision_score(yt,yp,average='macro',zero_division=0),4),
                    round(f1_score(yt,yp,average='macro',zero_division=0),4))

        a_tr,p_tr,f_tr = sm(ftr, y_train)
        a_v, p_v, f_v  = sm(fv,  y_val)
        a_te,p_te,f_te = sm(fte, y_test)

        results_svm.append({
            'Modelo':base_name,'Kernel':kernel,
            'C':clf.best_params_['C'],
            'gamma':clf.best_params_.get('gamma','-'),
            'Acc_Train':a_tr,'Acc_Val':a_v,'Acc_Test':a_te,
            'Prec_Train':p_tr,'Prec_Val':p_v,'Prec_Test':p_te,
            'F1_Train':f_tr,'F1_Val':f_v,'F1_Test':f_te
        })
        liberar_memoria(clf, best)

    liberar_memoria(ftr, fv, fte)

df_svm = pd.DataFrame(results_svm)
print('\n═══ TABLA 4 — SVM + FEATURES — MNIST ═══')
display(df_svm)

  Convirtiendo imágenes para VGG16...
  Extrayendo features...
  Shape: (48000, 512)

  ▶ SVM linear | VGG16
  Params óptimos: {'C': 0.01}

  ▶ SVM rbf | VGG16
  Params óptimos: {'C': 10, 'gamma': 0.001}
  Convirtiendo imágenes para ResNet50...
  Extrayendo features...
  Shape: (48000, 2048)

  ▶ SVM linear | ResNet50
  Params óptimos: {'C': 0.1}

  ▶ SVM rbf | ResNet50
  Params óptimos: {'C': 10, 'gamma': 'scale'}

═══ TABLA 4 — SVM + FEATURES — MNIST ═══


,Modelo,Kernel,C,gamma,Acc_Train,Acc_Val,Acc_Test,Prec_Train,Prec_Val,Prec_Test,F1_Train,F1_Val,F1_Test
0,VGG16,linear,0.01,-,0.9845,0.9702,0.9746,0.9843,0.9697,0.9743,0.9842,0.9697,0.9743
1,VGG16,rbf,10.00,0.001,0.9951,0.9717,0.9750,0.9950,0.9713,0.9749,0.9950,0.9713,0.9748
2,ResNet50,linear,0.10,-,0.9804,0.9643,0.9709,0.9802,0.9636,0.9706,0.9801,0.9636,0.9705
3,ResNet50,rbf,10.00,scale,0.9928,0.9652,0.9684,0.9928,0.9650,0.9685,0.9927,0.9647,0.9682


In [15]:
df_svm.to_csv('/kaggle/working/resultados_svm_mnist.csv', index=False)
print('Guardado ✓')

Guardado ✓
